# 01 — Explorative Datenanalyse

**Etappe 5.** Ziel dieses Notebooks ist nicht, hübsche Bilder zu erzeugen, sondern
Entscheidungen zu begründen, die in den folgenden Etappen getroffen werden:
welche Metrik sinnvoll ist, welche Wertebereiche in den Datenvertrag gehören,
welche Spalten entfernt werden müssen und wie die Daten aufgeteilt werden dürfen.

**Regel für dieses Notebook:** Nach jeder Abbildung steht eine Markdown-Zelle mit
*einem Satz* — was man sieht und was daraus folgt. Ein Notebook aus lauter
Diagrammen ohne Text ist wertlos, sobald man es vier Wochen später selbst liest.

Das Notebook muss mit `Kernel → Restart & Run All` fehlerfrei durchlaufen.

In [2]:
import numpy as np
import pandas as pd

from src.data.load import LEAKAGE_COLUMNS, TARGET_COLUMN, load_raw_data
from src.data.validate import load_params, validate_dataframe
from src.utils.plotting import (
    CMAP_KORRELATION,
    FARBEN,
    KLASSEN,
    nur_x_gitter,
    nur_y_gitter,
    setze_stil,
    speichere,
)

import matplotlib.pyplot as plt

setze_stil()
params = load_params()

# Die Rohdaten werden ueber load_raw_data() geladen: dort wird der Hash geprueft.
df = load_raw_data()
df.shape

ModuleNotFoundError: No module named 'src'

## 1 Der Datenvertrag als erster Blick

Bevor irgendetwas gezeichnet wird: Halten die Daten, was der Vertrag zusagt?
Das ist schneller als jede Abbildung und fängt die Fehler ab, die sonst still
durchlaufen.

In [ ]:
ergebnis = validate_dataframe(df)
print(ergebnis.report())

Der Vertrag ist ohne Beanstandung eingehalten — die Daten sind vollständig, alle
Werte liegen im physikalisch plausiblen Bereich und es treten keine unbekannten
Kategorien auf.

## 2 Überblick über Umfang und Typen

In [ ]:
uebersicht = pd.DataFrame({
    "Typ": df.dtypes.astype(str),
    "fehlend": df.isna().sum(),
    "eindeutige Werte": df.nunique(),
})
print(f"{df.shape[0]} Zeilen x {df.shape[1]} Spalten")
uebersicht

In [ ]:
merkmale_numerisch = [
    "air_temperature_k",
    "process_temperature_k",
    "rotational_speed_rpm",
    "torque_nm",
    "tool_wear_min",
]
df[merkmale_numerisch].describe().T.round(2)

Der Datensatz ist vollständig — keine fehlenden Werte in keiner Spalte — und die
fünf Prozessgrößen liegen in engen, physikalisch plausiblen Bereichen; eine
Bereinigung ist also nicht nötig, und der Aufwand kann in die Modellierung fließen.

## 3 Klassenverhältnis

Die wichtigste einzelne Zahl des Projekts: Sie bestimmt Metrik und
Entscheidungsschwelle.

In [ ]:
anzahl = df[TARGET_COLUMN].value_counts().sort_index()
anteil = df[TARGET_COLUMN].value_counts(normalize=True).sort_index()
positiv_anteil = float(anteil.loc[1])

print(f"kein Ausfall : {anzahl.loc[0]:>5d}  ({anteil.loc[0]:.2%})")
print(f"Ausfall      : {anzahl.loc[1]:>5d}  ({anteil.loc[1]:.2%})")
print(f"\nAccuracy der Vorhersage 'faellt nie aus': {anteil.loc[0]:.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.2))

positionen = [0, 1]
werte = [anzahl.loc[0], anzahl.loc[1]]
farben = [FARBEN["kein_ausfall"], FARBEN["ausfall"]]

balken = ax.barh(positionen, werte, height=0.55, color=farben)

# Direkte Beschriftung statt Legende: eine Reihe, zwei Kategorien.
for pos, wert, anteil_wert in zip(positionen, werte, [anteil.loc[0], anteil.loc[1]]):
    ax.text(wert + 120, pos, f"{wert:,}".replace(",", ".") + f"  ({anteil_wert:.2%})",
            va="center", ha="left", fontsize=9.5, color=FARBEN["text"])

ax.set_yticks(positionen, [KLASSEN[0], KLASSEN[1]])
ax.set_xlim(0, 11500)
ax.set_xlabel("Anzahl Betriebszustände")
ax.set_title("Nur 3,4 % der Betriebszustände sind Ausfälle", loc="left")
nur_x_gitter(ax)
ax.invert_yaxis()

speichere(fig, "01_klassenverhaeltnis")
plt.show()

Nur 339 von 10.000 Betriebszuständen sind Ausfälle (3,39 %), weshalb **Accuracy als
Metrik ausscheidet**: die Vorhersage „fällt nie aus" erreicht damit 96,61 % und ist
trotzdem wertlos — bewertet wird stattdessen über PR-AUC und die erwarteten Kosten.

## 4 Verteilungen der Prozessgrößen nach Zielklasse

Weil die Klassen so ungleich besetzt sind, werden die Häufigkeiten je Klasse auf
1 normiert — sonst wäre die Ausfallklasse im Diagramm unsichtbar.

In [ ]:
fig, achsen = plt.subplots(2, 3, figsize=(13, 6.5))
achsen = achsen.ravel()

beschriftung = {
    "air_temperature_k": "Lufttemperatur [K]",
    "process_temperature_k": "Prozesstemperatur [K]",
    "rotational_speed_rpm": "Drehzahl [1/min]",
    "torque_nm": "Drehmoment [Nm]",
    "tool_wear_min": "Werkzeugverschleiß [min]",
}

for ax, spalte in zip(achsen, merkmale_numerisch):
    grenzen = np.linspace(df[spalte].min(), df[spalte].max(), 40)
    for klasse, farbe in [(0, FARBEN["kein_ausfall"]), (1, FARBEN["ausfall"])]:
        ax.hist(
            df.loc[df[TARGET_COLUMN] == klasse, spalte],
            bins=grenzen, density=True, histtype="stepfilled",
            color=farbe, alpha=0.45 if klasse == 0 else 0.75,
            edgecolor=farbe, linewidth=1.2, label=KLASSEN[klasse],
        )
    ax.set_title(beschriftung[spalte], loc="left", fontsize=10.5)
    ax.set_yticks([])
    nur_y_gitter(ax)
    ax.grid(False)

achsen[-1].axis("off")
# Legende in das freie sechste Feld statt ueber die Balken.
griffe, namen = achsen[0].get_legend_handles_labels()
achsen[-1].legend(griffe, namen, loc="center left", fontsize=11)

fig.suptitle("Verteilungen je Zielklasse, je Klasse auf 1 normiert",
             x=0.005, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout()

speichere(fig, "02_verteilungen_nach_klasse")
plt.show()

Am deutlichsten trennen **Drehmoment** und **Werkzeugverschleiß** — Ausfälle liegen
bei hoher Last und bei Verschleiß oberhalb von rund 200 Minuten —, während die
Drehzahl Ausfälle an *beiden* Rändern zeigt und die Temperaturen sich nur leicht
nach oben verschieben; keine Einzelgröße trennt also sauber, entscheidend ist ihr
Zusammenspiel (Abschnitt 8).

## 5 Qualitätsvariante und Ausfallrate

In [ ]:
je_typ = (
    df.groupby("type")[TARGET_COLUMN]
    .agg(anzahl="size", ausfaelle="sum")
    .reindex(["L", "M", "H"])
)
je_typ["ausfallrate"] = je_typ["ausfaelle"] / je_typ["anzahl"]
je_typ

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.2))

bezeichnung = {"L": "L — niedrige Qualität", "M": "M — mittlere", "H": "H — hohe"}
positionen = np.arange(len(je_typ))

ax.bar(positionen, je_typ["ausfallrate"], width=0.5, color=FARBEN["kein_ausfall"])

for pos, rate in zip(positionen, je_typ["ausfallrate"]):
    ax.text(pos, rate + 0.0012, f"{rate:.2%}", ha="center", va="bottom",
            fontsize=9.5, color=FARBEN["text"])

ax.axhline(positiv_anteil, color=FARBEN["neutral"], linewidth=1.2, linestyle="--")
ax.text(len(je_typ) - 0.4, positiv_anteil + 0.0012, "Gesamtrate 3,39 %",
        ha="right", va="bottom", fontsize=8.5, color=FARBEN["text_sekundaer"])

ax.set_xticks(positionen, [bezeichnung[t] for t in je_typ.index])
ax.set_ylabel("Ausfallrate")
ax.set_ylim(0, je_typ["ausfallrate"].max() * 1.35)
ax.yaxis.set_major_formatter(lambda x, _: f"{x:.1%}")
ax.set_title("Niedrige Qualitätsvariante fällt am häufigsten aus", loc="left")
nur_y_gitter(ax)

speichere(fig, "03_ausfallrate_je_typ")
plt.show()

Die Ausfallrate steigt von der hohen zur niedrigen Qualitätsvariante spürbar an,
weshalb `type` als Merkmal ins Modell gehört — allerdings one-hot-kodiert, da die
drei Stufen zwar geordnet sind, ihr Abstand aber nicht bezifferbar ist.

## 6 Zusammenhänge zwischen den Prozessgrößen

In [ ]:
spalten = merkmale_numerisch + [TARGET_COLUMN]
korrelation = df[spalten].corr()

etikett = {**beschriftung, TARGET_COLUMN: "Ausfall (Ziel)"}
namen = [etikett[s] for s in spalten]

fig, ax = plt.subplots(figsize=(7.2, 6))
bild = ax.imshow(korrelation, cmap=CMAP_KORRELATION, vmin=-1, vmax=1)

ax.set_xticks(range(len(namen)), namen, rotation=35, ha="right")
ax.set_yticks(range(len(namen)), namen)
ax.grid(False)

for i in range(len(namen)):
    for j in range(len(namen)):
        wert = korrelation.iloc[i, j]
        ax.text(j, i, f"{wert:.2f}", ha="center", va="center", fontsize=8.5,
                color="white" if abs(wert) > 0.55 else FARBEN["text"])

leiste = fig.colorbar(bild, ax=ax, shrink=0.75)
leiste.outline.set_visible(False)
leiste.set_label("Pearson-Korrelation", fontsize=9)

ax.set_title("Drehzahl und Drehmoment sind stark gegenläufig", loc="left")
fig.tight_layout()

speichere(fig, "04_korrelationen")
plt.show()

Drehzahl und Drehmoment korrelieren stark negativ (die mechanische Leistung ist
im Datensatz näherungsweise konstant), und die beiden Temperaturen hängen eng
zusammen — für baumbasierte Modelle ist das unkritisch, für die logistische
Regression in Etappe 7 ist es beim Interpretieren der Koeffizienten zu beachten.

## 7 Nachweis des Data Leakage

Die Spalten `TWF`, `HDF`, `PWF`, `OSF` und `RNF` beschreiben die **Ursache** eines
Ausfalls. Sie entstehen gemeinsam mit dem Ausfall und sind zum Vorhersagezeitpunkt
nicht bekannt. Hier wird belegt, warum sie raus müssen — der gemessene Unterschied
in der Modellgüte folgt in Etappe 6.

In [ ]:
ursachen = pd.DataFrame({
    "Faelle": [int(df[c].sum()) for c in LEAKAGE_COLUMNS],
    "Korrelation mit Ziel": [float(df[c].corr(df[TARGET_COLUMN])) for c in LEAKAGE_COLUMNS],
}, index=[c.upper() for c in LEAKAGE_COLUMNS])

# Wie viele Ausfaelle werden allein durch die Ursachenspalten erklaert?
erklaert = (df[list(LEAKAGE_COLUMNS)].sum(axis=1) > 0)
print(f"Ausfaelle gesamt                     : {int(df[TARGET_COLUMN].sum())}")
print(f"davon mit gesetzter Ursachenspalte   : {int((erklaert & (df[TARGET_COLUMN] == 1)).sum())}")
ursachen.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))

positionen = np.arange(len(ursachen))
werte = ursachen["Korrelation mit Ziel"].to_numpy()

ax.bar(positionen, werte, width=0.5, color=FARBEN["ausfall"])
for pos, wert in zip(positionen, werte):
    ax.text(pos, wert + 0.012, f"{wert:.2f}", ha="center", va="bottom",
            fontsize=9.5, color=FARBEN["text"])

ax.set_xticks(positionen, ursachen.index)
ax.set_ylabel("Korrelation mit der Zielgröße")
ax.set_ylim(0, max(werte) * 1.3)
ax.set_title("Die Ursachenspalten korrelieren auffällig stark mit dem Ziel", loc="left")
nur_y_gitter(ax)

speichere(fig, "05_leakage_ursachenspalten")
plt.show()

Die Ursachenspalten erklären fast jeden Ausfall und korrelieren entsprechend stark
mit der Zielgröße, weshalb sie in Etappe 6 vor der Aufteilung entfernt werden —
ein Modell mit diesen Spalten sagt nicht den Ausfall vorher, sondern liest ihn ab.

## 8 Der Betriebspunkt: Drehmoment gegen Drehzahl

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2))

normal = df[df[TARGET_COLUMN] == 0]
ausfall = df[df[TARGET_COLUMN] == 1]

ax.scatter(normal["rotational_speed_rpm"], normal["torque_nm"],
           s=6, color=FARBEN["kein_ausfall"], alpha=0.18, linewidths=0,
           label=KLASSEN[0])
ax.scatter(ausfall["rotational_speed_rpm"], ausfall["torque_nm"],
           s=22, color=FARBEN["ausfall"], alpha=0.9,
           edgecolors="white", linewidths=0.6, label=KLASSEN[1])

ax.set_xlabel("Drehzahl [1/min]")
ax.set_ylabel("Drehmoment [Nm]")
ax.set_title("Ausfälle liegen an den Rändern des Betriebsbereichs", loc="left")
ax.legend(loc="upper right", markerscale=1.6)

speichere(fig, "06_betriebspunkt")
plt.show()

Die Ausfälle sammeln sich an zwei gegenüberliegenden Rändern — hohe Last bei
niedriger Drehzahl und niedrige Last bei hoher Drehzahl —, also in Bereichen
auffälliger mechanischer Leistung; genau diese Nichtlinearität wird ein lineares
Modell in Etappe 7 nicht abbilden können.

## 9 Gruppenstruktur — darf zeilenweise aufgeteilt werden?

Wenn mehrere Zeilen zur selben Maschine gehören, dürfen sie nicht gleichzeitig in
Trainings- und Testmenge landen. Das ist vor Etappe 6 zu klären.

In [ ]:
print("Zeilen gesamt          :", len(df))
print("eindeutige product_id  :", df["product_id"].nunique())
print("eindeutige udi         :", df["udi"].nunique())
print("\nJede Zeile ein eigenes Werkstueck:", bool(df["product_id"].is_unique))

Jede Zeile trägt eine eigene `product_id`, es gibt also keine Gruppenstruktur und
keinen Zeitstempel — die Aufteilung in Etappe 6 darf zeilenweise erfolgen,
geschichtet nach der Zielklasse (`stratify`), und braucht weder `GroupShuffleSplit`
noch einen zeitlichen Schnitt.

## 10 Vorschau: physikalisch abgeleitete Größen

Fachwissen statt Ausprobieren. Zwei Größen drängen sich auf:

* **Temperaturdifferenz** `process_temperature_k − air_temperature_k` — wie gut die
  Wärme abgeführt wird.
* **Mechanische Leistung** `P = M · 2πn / 60` — die Größe, die Drehmoment und
  Drehzahl zusammenfasst.

Gebaut werden sie in Etappe 8, und zwar **innerhalb** der sklearn-Pipeline. Hier
wird nur geprüft, ob sie überhaupt trennen.

In [ ]:
vorschau = pd.DataFrame({
    "temp_difference_k": df["process_temperature_k"] - df["air_temperature_k"],
    "power_w": df["torque_nm"] * df["rotational_speed_rpm"] * 2 * np.pi / 60,
    TARGET_COLUMN: df[TARGET_COLUMN],
})

fig, achsen = plt.subplots(1, 2, figsize=(11, 4))

for ax, spalte, titel in zip(
    achsen,
    ["temp_difference_k", "power_w"],
    ["Temperaturdifferenz [K]", "Mechanische Leistung [W]"],
):
    daten = [vorschau.loc[vorschau[TARGET_COLUMN] == k, spalte] for k in (0, 1)]
    kasten = ax.boxplot(daten, patch_artist=True, widths=0.45, showfliers=False,
                        medianprops={"color": "white", "linewidth": 1.6})
    for koerper, farbe in zip(kasten["boxes"],
                              [FARBEN["kein_ausfall"], FARBEN["ausfall"]]):
        koerper.set_facecolor(farbe)
        koerper.set_edgecolor(farbe)
    for teil in ["whiskers", "caps"]:
        for linie in kasten[teil]:
            linie.set_color(FARBEN["neutral"])
    ax.set_xticks([1, 2], [KLASSEN[0], KLASSEN[1]])
    ax.set_title(titel, loc="left", fontsize=10.5)
    nur_y_gitter(ax)

fig.suptitle("Abgeleitete Größen trennen die Klassen deutlich",
             x=0.005, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout()

speichere(fig, "07_abgeleitete_groessen")
plt.show()

Beide abgeleiteten Größen trennen die Klassen deutlich besser als die
Einzelmerkmale — die Ausfälle liegen bei geringerer Temperaturdifferenz und bei
auffälliger Leistung —, was den Aufwand der Merkmalskonstruktion in Etappe 8
rechtfertigt, bevor überhaupt an Hyperparameter gedacht wird.

## Zusammenfassung — was daraus für die folgenden Etappen folgt

| Befund | Folge |
|---|---|
| 3,39 % positive Fälle | Accuracy scheidet aus; PR-AUC und erwartete Kosten als Metrik (Etappe 1) |
| Keine fehlenden Werte, alle Werte plausibel | Keine Bereinigung nötig; der Datenvertrag aus Etappe 4 hält |
| Ursachenspalten erklären fast jeden Ausfall | `TWF, HDF, PWF, OSF, RNF` vor der Aufteilung entfernen (Etappe 6) |
| Jede Zeile ein eigenes Werkstück, kein Zeitstempel | Zeilenweise, geschichtete Aufteilung zulässig (Etappe 6) |
| Ausfälle an den Rändern des Betriebsbereichs | Lineares Modell wird deutlich schlechter sein als ein Baumverfahren (Etappe 7 und 9) |
| Temperaturdifferenz und Leistung trennen gut | Merkmalskonstruktion in der Pipeline lohnt sich (Etappe 8) |
| Ausfallrate steigt von H über M nach L | `type` one-hot-kodiert als Merkmal aufnehmen (Etappe 8) |
| Drehzahl und Drehmoment stark gegenläufig | Bei der logistischen Regression beim Deuten der Koeffizienten beachten |

Alle Abbildungen liegen unter `reports/figures/`.